# 02 · Message Properties

Every Service Bus message carries:

- a **body** (binary, but commonly UTF-8 text or JSON)
- a set of **system properties** (`MessageId`, `CorrelationId`, `Subject`, `ContentType`, `TimeToLive`, ...)
- a dictionary of **application properties** — your custom key/value metadata (used for routing & filtering)

System properties enable broker features (deduplication, sessions, routing). Application properties are for **your** business logic and for subscription filters (see notebook 08).


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;
using System.Text.Json;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.QueueName);

## 1. Send a richly-decorated message

In [ ]:
var order = new { OrderId = 42, Customer = "acme", Total = 199.99m };
var body  = JsonSerializer.Serialize(order);

var msg = new ServiceBusMessage(body)
{
    MessageId       = order.OrderId.ToString(),
    CorrelationId   = "trace-abc-123",
    Subject         = "order.created",
    ContentType     = "application/json",
    TimeToLive      = TimeSpan.FromMinutes(10),
    ApplicationProperties =
    {
        ["priority"] = "high",
        ["region"]   = "us-east",
        ["tenantId"] = "acme",
    }
};

await sender.SendMessageAsync(msg);
Console.WriteLine($"Sent {msg.MessageId} ({msg.Subject})");

## 2. Read it back and inspect every field

Notice how the receiver also sees **broker-assigned** properties such as `SequenceNumber`, `EnqueuedTime`, and `DeliveryCount`.


In [ ]:
var receiver = client.CreateReceiver(Config.QueueName);
var received = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));

Console.WriteLine($"MessageId       : {received.MessageId}");
Console.WriteLine($"CorrelationId   : {received.CorrelationId}");
Console.WriteLine($"Subject         : {received.Subject}");
Console.WriteLine($"ContentType     : {received.ContentType}");
Console.WriteLine($"SequenceNumber  : {received.SequenceNumber}");
Console.WriteLine($"EnqueuedTime    : {received.EnqueuedTime:o}");
Console.WriteLine($"DeliveryCount   : {received.DeliveryCount}");
Console.WriteLine($"TimeToLive      : {received.TimeToLive}");
Console.WriteLine("ApplicationProperties:");
foreach (var kv in received.ApplicationProperties)
    Console.WriteLine($"  {kv.Key} = {kv.Value}");
Console.WriteLine($"Body: {received.Body}");

await receiver.CompleteMessageAsync(received);

## 3. Duplicate detection (advanced)

If the queue is created with `RequiresDuplicateDetection = true`, the broker will reject any message whose `MessageId` matches one already enqueued within the detection window. This template doesn't enable it, but it's a one-line property when you do.

```csharp
// administration client
await admin.CreateQueueAsync(new CreateQueueOptions("dedupe-queue")
{
    RequiresDuplicateDetection      = true,
    DuplicateDetectionHistoryTimeWindow = TimeSpan.FromMinutes(10),
});
```


In [ ]:
await sender.DisposeAsync();
await receiver.DisposeAsync();
await client.DisposeAsync();

Next: [`03-batching.ipynb`](03-batching.ipynb)